In [55]:
from huggingface_hub import HfApi

# 클래스 - HubAPI 클라이언트
api = HfApi()

# 인스턴스 메서드 - ModelInfo
info = api.model_info(
  repo_id="HuggingFaceTB/SmolLM2-135M-Instruct"
)

print(type(info))
print(info.id)
print(info.sha)
print(info.pipeline_tag)

<class 'huggingface_hub.hf_api.ModelInfo'>
HuggingFaceTB/SmolLM2-135M-Instruct
12fd25f77366fa6b3b4b768ec3050bf629380bac
text-generation


In [56]:
# info객체에 들어있는 파일들 종류
for file in info.siblings:
  print(file.rfilename)

.gitattributes
README.md
all_results.json
config.json
eval_results.json
generation_config.json
merges.txt
model.safetensors
onnx/model.onnx
onnx/model_bnb4.onnx
onnx/model_fp16.onnx
onnx/model_int8.onnx
onnx/model_q4.onnx
onnx/model_q4f16.onnx
onnx/model_quantized.onnx
onnx/model_uint8.onnx
runs/Oct31_10-14-22_ip-26-0-174-36/events.out.tfevents.1730370128.ip-26-0-174-36.3239327.0
runs/Oct31_10-14-22_ip-26-0-174-36/events.out.tfevents.1730376056.ip-26-0-174-36.3239327.1
special_tokens_map.json
tokenizer.json
tokenizer_config.json
train_results.json
trainer_state.json
training_args.bin
vocab.json


In [57]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
  repo_id="HuggingFaceTB/SmolLM2-135M-Instruct",
  filename="config.json",
  revision=info.sha # 특정 커밋 버전 파일
)

print(type(file_path))
print(file_path)

<class 'str'>
C:\Users\hurwa\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-135M-Instruct\snapshots\12fd25f77366fa6b3b4b768ec3050bf629380bac\config.json


In [58]:
import json
from pathlib import Path

config_data = json.loads(
  Path(file_path).read_text(encoding="utf-8")
)

print(type(config_data))
print(config_data["model_type"])

<class 'dict'>
llama


In [59]:
from transformers import AutoConfig

# 모델 설정을 읽고, model_type 등에 따라 적절한 PreTrainedConfig를 만들어내기
config = AutoConfig.from_pretrained(
  "HuggingFaceTB/SmolLM2-135M-Instruct",
  revision=info.sha
)

print(type(config))
print(config.model_type)
print(config.hidden_size)
print(config.num_hidden_layers)

<class 'transformers.models.llama.configuration_llama.LlamaConfig'>
llama
576
30


In [60]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
  "HuggingFaceTB/SmolLM2-135M-Instruct",
  revision=info.sha
)

messages = [
  {"role": "user", "content": "What is Hugging Face?"}
]

inputs = tokenizer.apply_chat_template(
  messages,
  add_generation_prompt=True,
  tokenize=True,
  return_dict=True,
  return_tensors="pt"
)

print(type(inputs))
print(inputs.keys())
print(inputs["input_ids"].shape)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
KeysView({'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198,  1780,   314,   407, 19712,  8182,    47,
             2,   198,     1,   520,  9531,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
torch.Size([1, 36])


In [61]:
import inspect

inspect.getdoc(HfApi.model_info)

'Get info on one specific model on huggingface.co\n\nModel can be private if you pass an acceptable token or are logged in.\n\nArgs:\n    repo_id (`str`):\n        A namespace (user or an organization) and a repo name separated\n        by a `/`.\n    revision (`str`, *optional*):\n        The revision of the model repository from which to get the\n        information.\n    timeout (`float`, *optional*):\n        Whether to set a timeout for the request to the Hub.\n    securityStatus (`bool`, *optional*):\n        Whether to retrieve the security status from the model\n        repository as well. The security status will be returned in the `security_repo_status` field.\n    files_metadata (`bool`, *optional*):\n        Whether or not to retrieve metadata for files in the repository\n        (size, LFS metadata, etc). Defaults to `False`.\n    expand (`list[ExpandModelProperty_T]`, *optional*):\n        List properties to return in the response. When used, only the properties in the li

In [62]:
# AutoTokenizer
from transformers import AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

# 1. tokenizer_config.json 확인
# 2. config.json 등 설정 확인
# 3. 사용할 Tokenizer 클래스 결정
# 4. tokenizer.json 등 필요한 파일 로드
# 5. 해당 Tokenizer 인스턴스 반환
tokenizer = AutoTokenizer.from_pretrained(
  MODEL_ID,
  use_fast=True
)

print(type(tokenizer))
print(tokenizer.__class__.__name__)
print(tokenizer.vocab_size)
print(len(tokenizer))

<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
GPT2Tokenizer
49152
49152


In [63]:
tokenizer("Hello")

{'input_ids': [19556], 'attention_mask': [1]}

In [64]:
tokenizer(
  "hello world",
  padding=False,
  truncation=None,
  max_length=None,
  return_tensors=None,
  return_attention_mask=None,
  add_special_tokens=True
)

{'input_ids': [28120, 905], 'attention_mask': [1, 1]}

In [65]:
a = tokenizer("Hello")
b = tokenizer(["Hello", "Good morning"])
c = tokenizer(
  ["Hello", "Good morning"],
  padding=True,
  return_tensors="pt"
)

for tmp in (a, b, c):
  print(type(tmp))
  for k, v in tmp.items():
    print("    ", k, type(v))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
     input_ids <class 'list'>
     attention_mask <class 'list'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
     input_ids <class 'list'>
     attention_mask <class 'list'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
     input_ids <class 'torch.Tensor'>
     attention_mask <class 'torch.Tensor'>


In [66]:
tokenizer

GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-135M-Instruct', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstr

In [67]:
text = "Hello Hugging Face!"

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print("tokens:", tokens)
print("ids:", ids)
print("decoded:", tokenizer.decode(ids))

tokens: ['Hello', 'ĠH', 'ugging', 'ĠFace', '!']
ids: [19556, 407, 19712, 8182, 17]
decoded: Hello Hugging Face!


In [68]:
tokens = tokenizer.tokenize(text)

ids = tokenizer.convert_tokens_to_ids(tokens)

print(list(zip(tokens, ids)))

[('Hello', 19556), ('ĠH', 407), ('ugging', 19712), ('ĠFace', 8182), ('!', 17)]


In [69]:
inputs = tokenizer(
  ["Hello", "How are you today?"],
  padding=True,
  return_tensors="pt"
)

In [70]:
# AutoTokenizer -> GPT2Tokenizer
print(type(tokenizer))

# input_ids, attention_mask -> KeyView
print(inputs.keys())
# tensor
print(type(inputs["input_ids"]))
# [1, N] -> [2, N]
print(inputs["input_ids"].shape)
# float32 -> torch.int64
print(inputs["input_ids"].dtype)

print(inputs["input_ids"].device)

<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
KeysView({'input_ids': tensor([[19556,     2,     2,     2,     2],
        [ 2020,   359,   346,  1834,    47]]), 'attention_mask': tensor([[1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1]])})
<class 'torch.Tensor'>
torch.Size([2, 5])
torch.int64
cpu


In [71]:
import torch
from transformers import (
  AutoTokenizer,
  AutoModelForCausalLM
)

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

device = torch.device(
  "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

messages = [
  {
    "role": "user",
    "content": "What is a tokenizer?"
  }
]

inputs = tokenizer.apply_chat_template(
  messages,
  tokenize=True,
  add_generation_prompt=True,
  return_dict=True,
  return_tensors="pt"
).to(device)

print("inputs:", type(inputs))
print("IDs:", inputs["input_ids"].shape)

tokenizer.decode(inputs["input_ids"])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

inputs: <class 'transformers.tokenization_utils_base.BatchEncoding'>
IDs: torch.Size([1, 35])


['<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is a tokenizer?<|im_end|>\n<|im_start|>assistant\n']

In [72]:
with torch.inference_mode():
  # CausalLMOutputWithPast
  outputs = model(
    **inputs,
    use_cache=False,
    return_dict=True
  )

# vocab 점수들
logits = outputs.logits

print("Outputs:", type(outputs))
print("logits:", logits.shape)

Outputs: <class 'transformers.modeling_outputs.CausalLMOutputWithPast'>
logits: torch.Size([1, 35, 49152])


In [74]:
for s in tokenizer.decode(logits.argmax(dim=2)):
  print(s)

,
<|im_end|> are an helpful assistant assistant, "ilingka. who to airing Face to
<|im_start|>ass
Can is the good in How
<|im_start|>assistant
A


In [ ]:
for text in tokenizer.batch_decode(logits.argmax(dim=-1)):
  print(text)

,
<|im_end|> are an helpful assistant assistant, "ilingka. who to airing Face to
<|im_start|>ass
Can is the good in How
<|im_start|>assistant
A


In [ ]:
logits[:, -1, :].shape

torch.Size([1, 49152])

In [ ]:
assert inputs["input_ids"].dtype == torch.int32

AssertionError: 

In [ ]:
assert inputs["input_ids"].ndim == 2

In [ ]:
inputs["input_ids"].dtype

torch.int64

In [79]:
inputs = tokenizer(
  ["My name is hurwan", "hello"],
  padding=True,
  truncation=True,
  return_tensors="pt"
)
type(inputs)

transformers.tokenization_utils_base.BatchEncoding

In [80]:
inputs.tokens(0)

['My', 'Ġname', 'Ġis', 'Ġhur', 'wan']

In [82]:
type(model).__mro__

(transformers.models.llama.modeling_llama.LlamaForCausalLM,
 transformers.models.llama.modeling_llama.LlamaPreTrainedModel,
 transformers.modeling_utils.PreTrainedModel,
 torch.nn.modules.module.Module,
 transformers.modeling_utils.EmbeddingAccessMixin,
 transformers.modeling_utils.ModuleUtilsMixin,
 transformers.utils.hub.PushToHubMixin,
 transformers.integrations.peft.PeftAdapterMixin,
 transformers.distributed.mixin.DistributedMixin,
 transformers.generation.utils.GenerationMixin,
 transformers.generation.continuous_batching.continuous_api.ContinuousMixin,
 object)

In [84]:
model.model

LlamaModel(
  (embed_tokens): Embedding(49152, 576, padding_idx=2)
  (layers): ModuleList(
    (0-29): 30 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((576,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [85]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

In [86]:
model.model.embed_tokens

Embedding(49152, 576, padding_idx=2)

In [87]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers.js_config": {
    "kv_cache_dtype": {
      "fp16": "float16",
      "q4f16": "float16"
    }
  },
  "transformers_version": "5.20.2",
  "use_cache": true,
  "vocab_size": 49152
}

In [88]:
inputs

{'input_ids': tensor([[ 5965,  1462,   314,  8472,  9469],
        [28120,     2,     2,     2,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0]])}

In [102]:
inputs = tokenizer(
  ["My name is hurwan", "hello"],
  padding=True,
  truncation=True,
  padding_side="left",
  return_tensors="pt"
)
type(inputs)

transformers.tokenization_utils_base.BatchEncoding

In [103]:
result = model.generate(
  **inputs,
  max_new_tokens=200,
  do_sample=True,
  return_dict_in_generate=True,
  output_scores=True,
  # padding=True,
  # padding_side='left'
)
print(type(result))

<class 'transformers.generation.utils.GenerateDecoderOnlyOutput'>


In [105]:
len(result.scores)

200

In [106]:
type(result.scores)

tuple

In [108]:
for idx, score in enumerate(result.scores):
  print(idx, score[score != float("-inf")].shape)

0 torch.Size([108])
1 torch.Size([102])
2 torch.Size([102])
3 torch.Size([103])
4 torch.Size([103])
5 torch.Size([101])
6 torch.Size([100])
7 torch.Size([111])
8 torch.Size([104])
9 torch.Size([101])
10 torch.Size([107])
11 torch.Size([101])
12 torch.Size([105])
13 torch.Size([102])
14 torch.Size([103])
15 torch.Size([101])
16 torch.Size([102])
17 torch.Size([102])
18 torch.Size([105])
19 torch.Size([101])
20 torch.Size([106])
21 torch.Size([104])
22 torch.Size([101])
23 torch.Size([102])
24 torch.Size([102])
25 torch.Size([102])
26 torch.Size([107])
27 torch.Size([100])
28 torch.Size([108])
29 torch.Size([103])
30 torch.Size([101])
31 torch.Size([105])
32 torch.Size([104])
33 torch.Size([106])
34 torch.Size([100])
35 torch.Size([103])
36 torch.Size([106])
37 torch.Size([101])
38 torch.Size([103])
39 torch.Size([102])
40 torch.Size([106])
41 torch.Size([106])
42 torch.Size([104])
43 torch.Size([105])
44 torch.Size([104])
45 torch.Size([102])
46 torch.Size([105])
47 torch.Size([101])
48

In [109]:
type(model.generate(
  **inputs,
  max_new_tokens=20
))

torch.Tensor